In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, input_file_name

spark = (
    SparkSession.builder
    .appName("Medallion Bronze")
    .master("local[*]")
    .getOrCreate()
)

print("========== SPARK APPLICATION INFO ==========")

print("Application ID   :", spark.sparkContext.applicationId)
print("Application Name :", spark.sparkContext.appName)
print("Spark Version    :", spark.version)
print("Master           :", spark.sparkContext.master)
print("Web UI           :", spark.sparkContext.uiWebUrl)
print("Default Parallel :", spark.sparkContext.defaultParallelism)
print("Default Partitons :",spark.conf.get("spark.sql.shuffle.partitions"))

print("============================================")
spark

========== SPARK APPLICATION INFO ==========
Application ID   : local-1792929479707
Application Name : Medallion Bronze
Spark Version    : 3.5.0
Master           : local[*]
Web UI           : http://Arjun:4042
Default Parallel : 18
Default Partitons : 200


In [5]:
from pyspark.sql.functions import (
    col, trim, to_date, regexp_replace
)

bronze_df = spark.read.parquet(
    "D:/pyspark_vs_code/medallion_project_duckdb/data/bronze"
)

silver_df = (
    bronze_df
    .withColumn("row_id", col("row_id").cast("long"))
    .withColumn("order_id", col("order_id").cast("long"))
    .withColumn("order_date", to_date(col("order_date"), "MM-dd-yyyy"))
    .withColumn("sales", col("sales").cast("decimal(12,2)"))
    .withColumn("category", trim(col("category")))
    .withColumn("sub_category", trim(col("sub_category")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    .withColumn("payment_type", trim(col("payment_type")))
    .dropDuplicates()
)

silver_df.show(5)
silver_df.printSchema()

+------+----------+--------+------+--------------------+-------------+------------+----------+------------+--------------------+--------------------+
|row_id|order_date|order_id| sales|            category| sub_category|        city|     state|payment_type|      ingestion_time|         source_file|
+------+----------+--------+------+--------------------+-------------+------------+----------+------------+--------------------+--------------------+
|   289|2011-06-19| 4000674|155.55|Outdoor Play Equi...|       Slides|   Cleveland|      Ohio|      credit|2026-10-25 17:34:...|file:///C:/Users/...|
|   345|2011-12-03| 4001891| 67.44|         Team Sports|Team Handball|      Austin|     Texas|      credit|2026-10-25 17:34:...|file:///C:/Users/...|
|   348|2011-11-30| 4007138|168.53|        Water Sports|  Windsurfing|     Fremont|California|      credit|2026-10-25 17:34:...|file:///C:/Users/...|
|   423|2011-08-09| 4005498|  5.60|        Water Sports|      Surfing|Indianapolis|   Indiana|      

In [6]:
silver_df.write \
    .mode("overwrite") \
    .parquet("D:/pyspark_vs_code/medallion_project_duckdb/data/silver")